# Notebook 12: Baseline Retraining at 260px (Resolution-Matched Comparison)

## Objective
PneumoXNet (proposed model) trains at 260×260 resolution, but the original ResNet-50 and
ConvNeXt-Tiny baselines (Notebook 06, 07) were trained at 224×224. This creates an unfair
comparison. This notebook retrains both baselines at 260×260 so all models in the paper are
compared under identical input resolution.

## What this notebook does
1. Loads dataset at 260px (same preprocessing/augmentation as PneumoXNet)
2. Retrains ResNet-50 and ConvNeXt-Tiny with identical training config
3. Evaluates both on the test set and prints classification reports
4. Saves results to CSV for direct comparison with PneumoXNet

In [1]:
# ============================================================
# Cell 2: Import Required Libraries
# ============================================================

import time
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import (
    resnet50, ResNet50_Weights,
    convnext_tiny, ConvNeXt_Tiny_Weights
)

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Use GPU if available, otherwise fall back to CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Cell 2 : Libraries Imported Successfully")
print(f"Device : {DEVICE}")

Cell 2 : Libraries Imported Successfully
Device : cuda


## Configuration
Same hyperparameters as PneumoXNet's final training run (Notebook 09, 260px version) —
this is essential for a fair comparison. Only the model architecture changes between runs.

In [2]:
# ============================================================
# Cell 4: Configuration
# ============================================================

IMAGE_SIZE = 260          # matches PneumoXNet input resolution (was 224 in old baselines)
BATCH_SIZE = 16
EPOCHS = 40                # max epochs; early stopping will likely cut this short
LEARNING_RATE = 1e-4
NUM_CLASSES = 3
NUM_WORKERS = 0
CLASS_NAMES = ["BACTERIA", "NORMAL", "VIRUS"]

# Project paths — same structure as previous notebooks
PROJECT_ROOT = Path("/mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI")
DATASET_DIR = PROJECT_ROOT / "dataset" / "processed_dataset"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Cell 4 : Configuration Set")
print(f"Image Size : {IMAGE_SIZE}px  |  Epochs : {EPOCHS}  |  Batch Size : {BATCH_SIZE}")

Cell 4 : Configuration Set
Image Size : 260px  |  Epochs : 40  |  Batch Size : 16


## Dataset Loading (260px)
Same augmentation pipeline used for PneumoXNet's 260px training run — this keeps the
data-side treatment identical across all models, isolating architecture as the only
variable being tested.

In [3]:
# ============================================================
# Cell 6: Dataset and DataLoaders (260px)
# ============================================================

# Training augmentation — random transforms to reduce overfitting
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.90, 1.10)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.10)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation/Test — no augmentation, only resize + normalize
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets from the same processed_dataset folder used throughout the project
train_dataset = datasets.ImageFolder(DATASET_DIR / "train", transform=train_transform)
valid_dataset = datasets.ImageFolder(DATASET_DIR / "validation", transform=eval_transform)
test_dataset  = datasets.ImageFolder(DATASET_DIR / "test", transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Class weights to handle imbalance (BACTERIA has more samples than NORMAL/VIRUS)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_dataset.targets),
    y=train_dataset.targets
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print("Cell 6 : Dataset Loaded")
print(f"Train : {len(train_dataset)}  |  Valid : {len(valid_dataset)}  |  Test : {len(test_dataset)}")

Cell 6 : Dataset Loaded
Train : 4099  |  Valid : 878  |  Test : 879


## Training / Validation / Test Helper Functions
Standard PyTorch training loop functions, reused across both ResNet-50 and ConvNeXt-Tiny
runs so the training procedure is identical for both models.

In [4]:
# ============================================================
# Cell 8: Training, Validation, and Test Evaluation Functions
# ============================================================

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Runs one full pass over the training set and updates model weights."""
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def validate_one_epoch(model, dataloader, criterion, device):
    """Evaluates the model on the validation set without updating weights."""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total


def evaluate_test_set(model, dataloader, device, class_names):
    """Runs final evaluation on the held-out test set and returns accuracy, F1, and report."""
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return acc, macro_f1, report


print("Cell 8 : Training/Validation/Test Functions Ready")

Cell 8 : Training/Validation/Test Functions Ready


## Model Builders
Both models are loaded with ImageNet-pretrained weights and their final classification
layer is replaced with a 3-class output layer — matching the exact head structure used
in the original Notebook 06 (ResNet-50) and Notebook 07 (ConvNeXt-Tiny), so the only
thing changing between old and new runs is input resolution.

In [5]:
# ============================================================
# Cell 10: Model Builder Functions
# ============================================================

def build_resnet50():
    """ResNet-50 with ImageNet weights, classification head replaced for 3 classes."""
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, NUM_CLASSES)
    )
    return model


def build_convnext_tiny():
    """ConvNeXt-Tiny with ImageNet weights, classification head replaced for 3 classes."""
    model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, NUM_CLASSES)
    return model


print("Cell 10 : Model Builders Ready (ResNet-50, ConvNeXt-Tiny)")

Cell 10 : Model Builders Ready (ResNet-50, ConvNeXt-Tiny)


## Main Training Runner
This function trains one model end-to-end: training loop with early stopping, saves the
best checkpoint (by validation accuracy), then loads that checkpoint and evaluates on the
test set. Used for both ResNet-50 and ConvNeXt-Tiny below so both follow the exact same
procedure.

In [6]:
# ============================================================
# Cell 12: Reusable Baseline Training + Evaluation Runner
# ============================================================

def train_baseline(model_name, model_builder):

    print("=" * 70)
    print(f"Training {model_name} at {IMAGE_SIZE}px")
    print("=" * 70)

    model = model_builder().to(DEVICE)

    # Same loss/optimizer/scheduler settings as PneumoXNet's final 260px run
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=2e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=4, min_lr=1e-6
    )

    best_val_acc = 0.0
    best_epoch = 0
    early_stopping_patience = 10
    early_stopping_counter = 0

    best_model_path = MODELS_DIR / f"{model_name.lower().replace('-', '')}_260px_best.pth"

    history = {"epoch": [], "train_loss": [], "train_acc": [], "valid_loss": [], "valid_acc": []}

    start_time = time.time()

    # ---------------- Training Loop ----------------
    for epoch in range(EPOCHS):

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        valid_loss, valid_acc = validate_one_epoch(model, valid_loader, criterion, DEVICE)

        scheduler.step(valid_acc)   # reduce LR if valid accuracy plateaus

        # Log this epoch's results
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["valid_loss"].append(valid_loss)
        history["valid_acc"].append(valid_acc)

        # Save checkpoint only if this epoch improved on best validation accuracy so far
        if valid_acc > best_val_acc:
            best_val_acc = valid_acc
            best_epoch = epoch + 1
            early_stopping_counter = 0
            torch.save(model.state_dict(), best_model_path)
            marker = "  <-- best"
        else:
            early_stopping_counter += 1
            marker = ""

        print(f"Epoch [{epoch+1}/{EPOCHS}] Train Acc: {train_acc:.4f}  "
              f"Valid Acc: {valid_acc:.4f}  Valid Loss: {valid_loss:.4f}{marker}")

        # Stop early if no improvement for `early_stopping_patience` epochs
        if early_stopping_counter >= early_stopping_patience:
            print("Early stopping triggered.")
            break

    elapsed = (time.time() - start_time) / 60
    print(f"\nTraining complete. Best Valid Acc: {best_val_acc:.4f} (epoch {best_epoch}), "
          f"Time: {elapsed:.2f} min")

    # ---------------- Load Best Checkpoint & Test ----------------
    model.load_state_dict(torch.load(best_model_path))
    test_acc, test_f1, report = evaluate_test_set(model, test_loader, DEVICE, CLASS_NAMES)

    print("\n" + "=" * 70)
    print(f"{model_name} — Test Set Results (260px)")
    print("=" * 70)
    print(f"Test Accuracy : {test_acc:.4f}")
    print(f"Test Macro-F1 : {test_f1:.4f}")
    print("-" * 70)
    print(report)

    # Save training history for plotting later if needed
    pd.DataFrame(history).to_csv(
        RESULTS_DIR / f"{model_name.lower().replace('-', '')}_260px_history.csv", index=False
    )

    # Free GPU memory before training the next model
    del model
    torch.cuda.empty_cache()

    return {
        "model": model_name,
        "best_val_acc": best_val_acc,
        "test_acc": test_acc,
        "test_macro_f1": test_f1,
        "training_time_min": round(elapsed, 2)
    }


print("Cell 12 : Training Runner Ready")

Cell 12 : Training Runner Ready


## Run 1: ResNet-50 at 260px
This retrains ResNet-50 (previously only trained at 224px in Notebook 06) using the
identical resolution PneumoXNet uses. Expect this to take roughly 20–30 minutes depending
on GPU and how early the early-stopping triggers.